### Task: Data Exploration, Cleansing, Model Building, and Exporting

#### 1. Explore and Cleanse the Data
- Analyze the dataset to identify missing or inconsistent values.
- Remove outliers based on interquartile range (IQR) for `DepDelay` and `ArrDelay`:
    - **Lower Bound (Departure Delay):** `lower_bound_dep = -17.0`
    - **Upper Bound (Departure Delay):** `upper_bound_dep = 15.0`
    - **Lower Bound (Arrival Delay):** `lower_bound_arr = -37.0`
    - **Upper Bound (Arrival Delay):** `upper_bound_arr = 27.0`
- Filter the dataset to include only rows within these bounds.
- Ensure all columns have the correct data types and no missing values.

#### 2. Build and Train the Model
- Use the `RandomForestClassifier` to predict the likelihood of a flight being delayed (`DepDel15`).
- Features used for training:
    - `Year`, `Month`, `DayofMonth`, `DayOfWeek`, `OriginAirportID`, `DestAirportID`, `CRSDepTime`
- Target variable: `DepDel15`
- Split the data into training (`X_train`, `y_train`) and testing (`X_test`, `y_test`) sets.
- Train the model using the training set and evaluate its performance on the test set.

#### 3. Export the Model
- Save the trained model to a file named `random_forest_model.pkl` for future use.

#### 4. Create a New CSV File for Airports
- Extract the list of all airports and their associated IDs from the `airports` DataFrame.
- Save the data to a CSV file named `airports.csv` in the `data/` directory.

In [30]:
# Load the data from the CSV file
data = pd.read_csv('data/flights.csv')


# Replace null values with zero
data.fillna(0, inplace=True)

# Display the first few rows to verify
print(data.head())

   Year  Month  DayofMonth  DayOfWeek Carrier  OriginAirportID  \
0  2013      9          16          1      DL            15304   
1  2013      9          23          1      WN            14122   
2  2013      9           7          6      AS            14747   
3  2013      7          22          1      OO            13930   
4  2013      5          16          4      DL            13931   

              OriginAirportName  OriginCity OriginState  DestAirportID  \
0           Tampa International       Tampa          FL          12478   
1      Pittsburgh International  Pittsburgh          PA          13232   
2  Seattle/Tacoma International     Seattle          WA          11278   
3  Chicago O'Hare International     Chicago          IL          11042   
4         Norfolk International     Norfolk          VA          10397   

                            DestAirportName    DestCity DestState  CRSDepTime  \
0             John F. Kennedy International    New York        NY        1539

In [31]:
# Identify and eliminate outliers in DepDelay and ArrDelay using quantiles

# Define quantile thresholds
dep_delay_lower = data['DepDelay'].quantile(0.05)  # 5th percentile
dep_delay_upper = data['DepDelay'].quantile(0.95)  # 95th percentile
arr_delay_lower = data['ArrDelay'].quantile(0.05)  # 5th percentile
arr_delay_upper = data['ArrDelay'].quantile(0.95)  # 95th percentile

# Filter the dataset to remove outliers
df_flights = data[
    (data['DepDelay'] >= dep_delay_lower) & (data['DepDelay'] <= dep_delay_upper) &
    (data['ArrDelay'] >= arr_delay_lower) & (data['ArrDelay'] <= arr_delay_upper)
]

# Display the filtered dataset
print(f"Dataset after removing outliers: {df_flights.shape}")
df_flights.head()

Dataset after removing outliers: (233175, 20)


,Year,Month,DayofMonth,DayOfWeek,Carrier,OriginAirportID,OriginAirportName,OriginCity,OriginState,DestAirportID,DestAirportName,DestCity,DestState,CRSDepTime,DepDelay,DepDel15,CRSArrTime,ArrDelay,ArrDel15,Cancelled
0,2013,9,16,1,DL,15304,Tampa International,Tampa,FL,12478,John F. Kennedy International,New York,NY,1539,4,0.0,1824,13,0,0
1,2013,9,23,1,WN,14122,Pittsburgh International,Pittsburgh,PA,13232,Chicago Midway International,Chicago,IL,710,3,0.0,740,22,1,0
2,2013,9,7,6,AS,14747,Seattle/Tacoma International,Seattle,WA,11278,Ronald Reagan Washington National,Washington,DC,810,-3,0.0,1614,-7,0,0
3,2013,7,22,1,OO,13930,Chicago O'Hare International,Chicago,IL,11042,Cleveland-Hopkins International,Cleveland,OH,804,35,1.0,1027,33,1,0
4,2013,5,16,4,DL,13931,Norfolk International,Norfolk,VA,10397,Hartsfield-Jackson Atlanta International,Atlanta,GA,545,-1,0.0,728,-9,0,0


In [35]:
from sklearn.metrics import accuracy_score

# Step 3: Select features and target
X = data[['DayOfWeek', 'DestAirportID']]
y = data['DepDel15']

# Step 4: Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 5: Train the model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Step 6: Evaluate the model
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")


# Save the model to a file to import later into flask
import pickle
pickle.dump(model, open('data/model.pkl', 'wb'))

print("Model saved as random_forest_model.pkl")

Model Accuracy: 0.80
Model saved as random_forest_model.pkl


In [36]:
# Show odds flight will be delayed to Las Vegas on a Monday
model.predict_proba([[1, 12892]])

/usr/local/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


array([[0.80781728, 0.19218272]])

In [ ]:
# Get unique column values for origin airport and id and export to CSV
df_flights[['OriginAirportID', 'OriginAirportName', 'OriginState', 'OriginCity']].drop_duplicates().to_csv('data/airports.csv', index=False)
print("Airports data saved to 'data/airports.csv'")

Airports data saved to 'data/airports.csv'
